In [2]:
# conda create -n web_scraping python=3.11
# conda activate web_scraping 
# cd documents/github/data-science-python/000248489_hw1_2025_1  
# pip install -r requirements.txt  

In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import re
import pandas as pd


driver = webdriver.Chrome()
url = "https://www.bumeran.com.pe/empleos.html"
driver.get(url)
driver.maximize_window()
time.sleep(10)

driver.execute_script("document.body.style.zoom='100%'")


In [4]:
# --------- Fecha de publicación ------------------

fecha_publicacion = driver.find_element(By.NAME,'icon-light-calendar')
fecha_publicacion.click()

time.sleep(5)

## Antes de 15 días

filtro_fecha = driver.find_element(By.XPATH, "//button[contains(text(), 'Menor a 15 días')]")
filtro_fecha.click()

time.sleep(5)

# -------------- Área --------------

area = driver.find_element(By.NAME,'icon-light-cube')
area.click()

time.sleep(5)

## Tecnología, Sistemas y Telecomunicaciones 

filtro_area = driver.find_element(By.XPATH, "//button[contains(text(), 'Tecnología, Sistemas y Telecomunicaciones')]")
filtro_area.click()

time.sleep(5)


# --------- Subárea --------------

subarea = driver.find_element(By.NAME,'icon-light-layers')
subarea.click()

time.sleep(5)

## Programación

filtro_area = driver.find_element(By.XPATH, "//button[contains(text(), 'Programación')]")
filtro_area.click()

time.sleep(8)

# --------- Departamento ------------------

dep = driver.find_element(By.XPATH, "//button//i[@name='icon-light-location-pin']/..")
dep.click()

time.sleep(9)

## Lima

filtro_area = driver.find_element(By.XPATH, "//button[contains(text(), 'Lima')]")
filtro_area.click()

time.sleep(10)


# ----------- Carga horaria ---------------------

carga_horaria = driver.find_element(By.NAME,'icon-light-bookmark')
carga_horaria.click()

time.sleep(10)

## Full time

filtro_area = driver.find_element(By.XPATH, "//button[contains(text(), 'Full-time')]")
filtro_area.click()

time.sleep(5)

In [5]:
# -----------------STAGE 1--------------------------

In [6]:
# Esperar a que los botones carguen
WebDriverWait(driver, 10).until(
    EC.presence_of_all_elements_located((By.CLASS_NAME, "sc-zDqdV"))  # Clases relevantes de los botones
)

# Encontrar todos los elementos <a> que contienen los enlaces

botones = driver.find_elements(By.CLASS_NAME, "sc-zDqdV")

# Extraer los href y guardarlos en una lista
enlaces = [boton.get_attribute("href") for boton in botones]

time.sleep(10)

enlaces

['https://www.bumeran.com.pe/empleos/software-architect-talent-house-peru-s.a.c.-1116750633.html',
 'https://www.bumeran.com.pe/empleos/analista-programador-power-builder-junior-1116745817.html',
 'https://www.bumeran.com.pe/empleos/desarrollador-front-ios-summit-s.a.c-1116742593.html',
 'https://www.bumeran.com.pe/empleos/analista-programador-de-tecnologias-de-la-informacion-instituto-quimioterapico-s.a.-iqfarma-1116740821.html',
 'https://www.bumeran.com.pe/empleos/frontend-developer-react-maseka-consulting-s.a.c.-1116739942.html',
 'https://www.bumeran.com.pe/empleos/analista-programador-de-sistemas-hibrido-tecsup-1116737470.html',
 'https://www.bumeran.com.pe/empleos/analista-programador-nodejs-web-html-grupo-tawa-1116751516.html',
 'https://www.bumeran.com.pe/empleos/desarrollador-en-innovacion-y-automatizacion-csti-corp-1116751489.html',
 'https://www.bumeran.com.pe/empleos/desarrollador-devops-hibrido-csti-corp-1116751467.html',
 'https://www.bumeran.com.pe/empleos/analista-prog

In [7]:
# Configurar Selenium
driver = webdriver.Chrome()
driver.maximize_window()

# Lista para almacenar los datos extraídos
datos_empleos = []

# Función para extraer el distrito correctamente
def get_distrito(driver):
    xpaths = [
        "/html/body/div[1]/div/div[2]/div[2]/div/div[2]/div[1]/div[2]/div[1]/div[1]/div[2]/div/div[1]/div[1]/div[2]/div/div/li/a/h2",
        "/html/body/div[1]/div/div[2]/div[2]/div/div[2]/div[1]/div[2]/div[1]/div[1]/div[2]/div/div/div[1]/div[2]/div/div/li/a/h2",
    ]
    
    for xpath in xpaths:
        try:
            distrito_texto = WebDriverWait(driver, 5).until(
                EC.presence_of_element_located((By.XPATH, xpath))
            ).text
            
            # Extraer solo la primera parte antes de la coma
            distrito_limpio = distrito_texto.split(",")[0].strip()
            return distrito_limpio
        except:
            continue  # Si falla, intenta con el siguiente XPath
    
    return "Distrito no encontrado"


# -------- Ahora se creará el cuadro

for enlace in enlaces:
    try:
        driver.get(enlace)
        time.sleep(3)  # Esperar que la página cargue completamente
        
        # Título
        titulo = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.TAG_NAME, 'h1'))
        ).text

        # Distrito
        distrito_limpio = get_distrito(driver)

        # Modalidad
        modalidad_xpaths = [
            "//ul[contains(@class, 'modalidad')]//li/a/p",
            "//div[contains(@class, 'sc')]/ul/div/li/a/p",
        ]
        modalidad = "Modo no disponible"
        for xpath in modalidad_xpaths:
            try:
                modalidad = driver.find_element(By.XPATH, xpath).text
                break
            except:
                continue

        # Descripción
        try:
            descripcion_completa = driver.find_element(By.ID, "ficha-detalle").text
            descripcion = "\n".join(descripcion_completa.split("\n")[:descripcion_completa.find("Beneficios")])
        except:
            descripcion = "Descripción no disponible"

        # Agregar datos a la lista
        datos_empleos.append([titulo, descripcion, distrito_limpio, modalidad, enlace])

        print(f"Datos extraídos de: {enlace}")

    except Exception as e:
        print(f"Error en {enlace}: {e}")

# Cerrar el navegador
driver.quit()

# Crear un DataFrame y exportar a Excel
tabla_empleos = pd.DataFrame(datos_empleos, columns=["Título", "Descripción", "Distrito", "Modalidad", "Enlace"])
tabla_empleos.to_excel("Trabajos_Bumeran.xlsx", index=False, engine="openpyxl")

print("Todos los datos han sido guardados en Trabajos_Bumeran.xlsx")

Datos extraídos de: https://www.bumeran.com.pe/empleos/software-architect-talent-house-peru-s.a.c.-1116750633.html
Datos extraídos de: https://www.bumeran.com.pe/empleos/analista-programador-power-builder-junior-1116745817.html
Datos extraídos de: https://www.bumeran.com.pe/empleos/desarrollador-front-ios-summit-s.a.c-1116742593.html
Datos extraídos de: https://www.bumeran.com.pe/empleos/analista-programador-de-tecnologias-de-la-informacion-instituto-quimioterapico-s.a.-iqfarma-1116740821.html
Datos extraídos de: https://www.bumeran.com.pe/empleos/frontend-developer-react-maseka-consulting-s.a.c.-1116739942.html
Datos extraídos de: https://www.bumeran.com.pe/empleos/analista-programador-de-sistemas-hibrido-tecsup-1116737470.html
Datos extraídos de: https://www.bumeran.com.pe/empleos/analista-programador-nodejs-web-html-grupo-tawa-1116751516.html
Datos extraídos de: https://www.bumeran.com.pe/empleos/desarrollador-en-innovacion-y-automatizacion-csti-corp-1116751489.html
Datos extraídos 

In [10]:
tabla_empleos


,Título,Descripción,Distrito,Modalidad,Enlace
0,Software Architect,"Actualizado hace 1 hora\nSan Isidro, Lima, Per...",San Isidro,Híbrido,https://www.bumeran.com.pe/empleos/software-ar...
1,Analista Programador Power Builder Junior,"Múltiples vacantes\nActualizado ayer\nLima, Li...",Lima,Remoto,https://www.bumeran.com.pe/empleos/analista-pr...
2,DESARROLLADOR FRONT IOS,"Múltiples vacantes\nActualizado ayer\nComas, L...",Comas,Híbrido,https://www.bumeran.com.pe/empleos/desarrollad...
3,Analista Programador de Tecnologías de la info...,"Actualizado ayer\nEl Agustino, Lima, Peru\nIQF...",El Agustino,Presencial,https://www.bumeran.com.pe/empleos/analista-pr...
4,Frontend Developer React,Múltiples vacantes\nActualizado ayer\nPueblo L...,Pueblo Libre,Híbrido,https://www.bumeran.com.pe/empleos/frontend-de...
5,Analista Programador de Sistemas - Híbrido,"Actualizado hace 2 días\nSanta Anita, Lima, Pe...",Santa Anita,Híbrido,https://www.bumeran.com.pe/empleos/analista-pr...
6,Analista Programador - NodeJs/Web HTML,"Nuevo\nPublicado hace 23 minutos\nLima, Lima, ...",Lima,Presencial,https://www.bumeran.com.pe/empleos/analista-pr...
7,Desarrollador en Innovación y Automatización,"Nuevo\nPublicado hace 30 minutos\nLima, Lima, ...",Lima,Híbrido,https://www.bumeran.com.pe/empleos/desarrollad...
8,Desarrollador DevOps - Híbrido,"Nuevo\nPublicado hace 38 minutos\nLima, Lima, ...",Lima,Híbrido,https://www.bumeran.com.pe/empleos/desarrollad...
9,Analista Programador FRONT-END - PRESENCIAL,"Nuevo\nPublicado hace 48 minutos\nLima, Lima, ...",Lima,Presencial,https://www.bumeran.com.pe/empleos/analista-pr...


In [11]:
tabla_empleos.to_csv("Trabajos_Bumeran.csv", index=False, encoding="utf-8", sep="|")